In [ ]:
# Cell 1: Install arc-agi from competition wheels (offline, works in Phase A and B)
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')

In [ ]:
# Cell 2: Write VericodingAgent to /tmp/my_agent.py
agent_code = '''
import os, sys, random
import numpy as np

# Framework imports (available in Phase B via ARC-AGI-3-Agents)
from agents.agent import Agent, GameAction


class VericodingAgent(Agent):
    """
    MoonArc3 strategy wrapped in official Agent interface.
    Uses FrameData.frame (list[list[int]]) — NOT .grid.
    Falls back to random on any error to ensure score > 0.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._visited = {}   # hash -> count
        self._step = 0
        self._prev_grid = None
        print('[VericodingAgent] init OK')

    def choose_action(self, frames, latest_frame) -> GameAction:
        try:
            return self._strategy(frames, latest_frame)
        except Exception as e:
            print(f'[VericodingAgent] strategy error: {e} -> random fallback')
            return self._random_action(latest_frame)

    def _strategy(self, frames, latest_frame) -> GameAction:
        self._step += 1
        # FrameData.frame = list[list[int]] (serialized grid rows)
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        available = latest_frame.available_actions

        # State hash for cycle detection
        state_key = grid.tobytes()
        visit_count = self._visited.get(state_key, 0)
        self._visited[state_key] = visit_count + 1

        # --- DENSE EXPLORER STRATEGY ---
        # Score each available action by novelty (prefer unvisited states)
        if available and len(available) > 0:
            # Avoid repeated actions if we've been here before
            if visit_count > 2:
                # Explore: pick random unvisited-looking action
                action_type = random.choice(available)
            else:
                # Greedy: pick first available
                action_type = available[0]
        else:
            action_type = None

        action = GameAction(action_type)

        # Handle complex actions (click with coordinates)
        if action.is_complex():
            h, w = grid.shape if grid.ndim == 2 else (10, 10)
            # Find most common non-zero color for targeted click
            flat = grid.flatten()
            nonzero = flat[flat > 0]
            if len(nonzero) > 0:
                target_color = int(np.bincount(nonzero).argmax())
                ys, xs = np.where(grid == target_color)
                idx = self._step % len(ys)
                x, y = int(xs[idx]), int(ys[idx])
            else:
                x = random.randint(0, max(w-1, 0))
                y = random.randint(0, max(h-1, 0))
            action.set_data({'x': x, 'y': y})

        return action

    def _random_action(self, latest_frame) -> GameAction:
        available = getattr(latest_frame, 'available_actions', None) or []
        action_type = random.choice(available) if available else None
        action = GameAction(action_type)
        if action.is_complex():
            action.set_data({'x': random.randint(0,63), 'y': random.randint(0,63)})
        return action


def build_agent(*args, **kwargs):
    return VericodingAgent(*args, **kwargs)
'''

with open('/tmp/my_agent.py', 'w') as f:
    f.write(agent_code)
print('[Cell2] VericodingAgent written to /tmp/my_agent.py')

In [ ]:
# Cell 3: Phase B — Competition Rerun (gateway + framework)
import os, subprocess, shutil, sys

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell3] Phase B: competition rerun detected')

    # Wait for gateway sidecar
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)
    print('[Cell3] gateway ready')

    # Copy framework from competition data
    comp = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    fw_src = f'{comp}/ARC-AGI-3-Agents'
    fw_dst = '/kaggle/working/ARC-AGI-3-Agents'
    if os.path.exists(fw_src):
        shutil.copytree(fw_src, fw_dst, dirs_exist_ok=True)
    elif os.path.exists(f'{comp}/arc_agi_3_agents'):
        shutil.copytree(f'{comp}/arc_agi_3_agents', fw_dst, dirs_exist_ok=True)
    print(f'[Cell3] framework copied to {fw_dst}')

    # Install agent into framework
    agents_dir = f'{fw_dst}/agents'
    shutil.copy('/tmp/my_agent.py', f'{agents_dir}/my_agent.py')

    # Register VericodingAgent in framework __init__
    init_content = '''
from agents.my_agent import build_agent
AGENT_NAME = "vericoding"
'''
    with open(f'{agents_dir}/__init__.py', 'w') as f:
        f.write(init_content)

    # Run framework
    sys.path.insert(0, fw_dst)
    result = subprocess.run(
        [sys.executable, 'main.py', '--agent', 'vericoding'],
        cwd=fw_dst, capture_output=False
    )
    print(f'[Cell3] framework exit code: {result.returncode}')
else:
    print('[Cell3] Phase A: skipping (no competition rerun)')

In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by framework gateway')